In [5]:
import os
import json
from model.rag_4b import solve_question_4b,solve_math_question_4b

d:\User\ProjectGithub\hiepnguyenn-99\RAG-Solve-Math\appenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
with open('data/cleaned/test/giai_tich_1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
test_data_math = data["problems"]
print(f"Số lượng bài test: {len(test_data_math)}")

Số lượng bài test: 395


In [7]:
def solve_math_batch(questions):
    """
    Nhận list câu hỏi, trả về list đáp án bằng cách gọi solve_math_question_4b từng câu.
    Nếu gặp lỗi, trả về None cho câu đó.
    """
    answers = []
    for q in questions:
        try:
            result = solve_math_question_4b(q)
            ans = result if isinstance(result, str) else result[0]
        except Exception as e:
            ans = None
        answers.append(ans)
    return answers

In [8]:

def process_in_batches(questions, batch_size):
    """
    """
    all_answers = []
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        batch_answers = solve_math_batch(batch)
        all_answers.extend(batch_answers)
    return all_answers

TEST giải toán

In [ ]:
# Test giải toán trên dữ liệu test và ghi kết quả từng câu ngay sau khi xử lý
# batch_size = 32
batch_size = 4
questions = [item['question'] for item in test_data_math]
results_math = []
os.makedirs('Answer', exist_ok=True)
with open('Answer/giai_tich_1.json', 'w', encoding='utf-8') as f:
    for idx in range(0, len(questions), batch_size):
        batch = questions[idx:idx+batch_size]
        answers = process_in_batches(batch, batch_size=len(batch))
        for i, (question, answer) in enumerate(zip(batch, answers), idx+1):
            results_math.append({"index": i, "input": question, "output": answer})
            f.seek(0)
            json.dump(results_math, f, ensure_ascii=False, indent=2)
            f.truncate()
        print(f"Đã ghi đến câu thứ {idx+len(batch)} ra file Answer/giai_tich_1.json")

In [ ]:
# Kiểm tra lại file answer RAG đã ghi ra
with open('Answer/giai_tich_1.json', 'r', encoding='utf-8') as f:
    answer_rag_data = json.load(f)
print(f"Số lượng kết quả trong file answer RAG: {len(answer_rag_data)}")
print("Ví dụ kết quả đầu tiên:", answer_rag_data[0] if answer_rag_data else "Không có dữ liệu")